# Performance Analysis

Deep-dive analysis of strategy performance, trade attribution, and improvement opportunities.

**Learning objectives:**
- Analyze trade-level performance
- Perform attribution analysis
- Identify performance patterns
- Find areas for improvement

**Time**: 25 minutes

In [ ]:
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))

## Load Trade History

Load detailed trade logs from a backtest run.

In [ ]:
# Load most recent backtest results
# In practice, load from state/backtests/[timestamp]/trades.json
trades_path = Path.cwd().parent / "state" / "backtests" / "latest" / "trades.json"

# Placeholder data
trades = [
    {"pnl": 5.2, "edge": 0.08, "slippage_bps": 25, "order_type": "limit"},
    {"pnl": -2.1, "edge": 0.06, "slippage_bps": 60, "order_type": "market"},
    {"pnl": 8.5, "edge": 0.12, "slippage_bps": 10, "order_type": "limit"},
    # ... more trades
]

df = pd.DataFrame(trades)
print(f"Loaded {len(df)} trades")
df.head()

## Trade-Level Analysis

In [ ]:
# Winners vs Losers
winners = df[df['pnl'] > 0]
losers = df[df['pnl'] <= 0]

print("Winner characteristics:")
print(f"  Count: {len(winners)}")
print(f"  Avg PnL: ${winners['pnl'].mean():.2f}")
print(f"  Avg edge: {winners['edge'].mean():.3f}")
print(f"  Avg slippage: {winners['slippage_bps'].mean():.1f} bps")

print("\nLoser characteristics:")
print(f"  Count: {len(losers)}")
print(f"  Avg PnL: ${losers['pnl'].mean():.2f}")
print(f"  Avg edge: {losers['edge'].mean():.3f}")
print(f"  Avg slippage: {losers['slippage_bps'].mean():.1f} bps")

## Attribution Analysis

Attribute PnL to different factors.

In [ ]:
# PnL by order type
by_order_type = df.groupby('order_type')['pnl'].agg(['sum', 'mean', 'count'])
print("PnL by order type:")
print(by_order_type)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

by_order_type['sum'].plot(kind='bar', ax=axes[0], title='Total PnL by Order Type')
axes[0].set_ylabel('Total PnL ($)')
axes[0].grid(alpha=0.3)

by_order_type['mean'].plot(kind='bar', ax=axes[1], title='Mean PnL by Order Type', color='orange')
axes[1].set_ylabel('Mean PnL ($)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Identify Improvement Opportunities

Find patterns that suggest areas for improvement.

In [ ]:
# Correlation between edge and PnL
correlation = df['edge'].corr(df['pnl'])
print(f"Edge-PnL correlation: {correlation:.3f}")

plt.figure(figsize=(10, 6))
plt.scatter(df['edge'], df['pnl'], alpha=0.5)
plt.xlabel('Edge')
plt.ylabel('PnL ($)')
plt.title('Edge vs PnL')
plt.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.grid(alpha=0.3)
plt.show()

if correlation > 0.5:
    print("Strong positive correlation: Edge is predictive of PnL")
elif correlation < 0.2:
    print("Weak correlation: Consider improving edge quality or execution")

## Recommendations

Based on analysis, generate improvement suggestions.

In [ ]:
print("Improvement Suggestions:")

# Check slippage
avg_slippage = df['slippage_bps'].mean()
if avg_slippage > 50:
    print(f"1. HIGH SLIPPAGE ({avg_slippage:.1f} bps avg)")
    print("   → Use more limit orders (increase aggressive_edge threshold)")

# Check order type performance
if 'market' in by_order_type.index and 'limit' in by_order_type.index:
    if by_order_type.loc['limit', 'mean'] > by_order_type.loc['market', 'mean']:
        print("2. LIMIT ORDERS OUTPERFORM")
        print("   → Consider using limit orders more frequently")

# Check win rate
win_rate = len(winners) / len(df)
if win_rate < 0.5:
    print(f"3. LOW WIN RATE ({win_rate:.1%})")
    print("   → Increase min_edge threshold or improve forecast quality")

print("\nNext Steps:")
print("- Implement suggested improvements")
print("- Re-run backtest")
print("- Compare before/after metrics")

## Next Steps

- Try notebook `04_parameter_tuning.ipynb` to optimize parameters
- Read `LEARNING.md` for systematic improvement techniques
- Read `BACKTESTING.md` for validation methodology